#### THIRD ATTEMPT

## Data Exploration

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.

**Chapter goal:**

The goal of this chapter is to explore, clean and prepare the labelled OpTC telemetry for the ML/DL models. I first explored the data to understand its structure and class distribution, then checked for issues such as missing, duplicate and infinite values. Since the complete dataset contains over 59 million events, I created a smaller and more computationally manageable subset using selected attack and control hosts. The events in this subset were kept in chronological order so that the same data can be reused for both the initial attack detection exercise and the eventual attack prediction task. After that, I removed highly missing and identifier columns, then saved the resulting cleaned sample as a Parquet file so it can be loaded directly for the next stage without repeating the data preparation process.

In [1]:
# Imports
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Libraries for models
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Libraries for evaluation
from sklearn import metrics

# Libraries for deep learning
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam


from pathlib import Path
import pyarrow.parquet as pq


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")





In [2]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Load aand preview the data

In [3]:
# Location of ready flattened parquet files in Google Drive
data_path = Path("/content/drive/MyDrive/solutions/OpTC_ready")

# Get all parquet files
files = list(data_path.glob("*.parquet"))

print(f"Number of files: {len(files)}")

# Inspect one
sample = pd.read_parquet(files[0])

print(sample.shape)
sample.head()
# sample.info()


Number of files: 41
(143207, 62)


,action,actorID,hostname,id,object,objectID,pid,ppid,principal,tid,...,task_process_uuid,tgt_pid,tgt_pid_uuid,tgt_tid,type,user,user_name,user_stack_base,user_stack_limit,value
0,START,8119de7c-c6d0-4c68-89ea-a70511b2b48d,SysClient0060.systemia.com,f6f51a04-210e-46dd-a738-06b979d65505,FLOW,2fd2eb0c-8580-49aa-8ef2-fddbc3104526,4,-1,,-1,...,None,None,None,None,None,None,None,None,None,None
1,START,8119de7c-c6d0-4c68-89ea-a70511b2b48d,SysClient0060.systemia.com,5c52aa2a-462d-48d1-9bce-5c22f4fd131d,FLOW,96cda055-a614-49ad-9c22-7f44ee51f6d1,4,-1,,-1,...,None,None,None,None,None,None,None,None,None,None
2,START,47573cec-2848-4e70-8e57-39a2416328d2,SysClient0060.systemia.com,ea950653-200d-4fcf-9352-213c8d2cdb03,FLOW,e6ce56f2-30b1-4b2e-8f96-e6f6b19f851f,2228,-1,,-1,...,None,None,None,None,None,None,None,None,None,None
3,START,8119de7c-c6d0-4c68-89ea-a70511b2b48d,SysClient0060.systemia.com,75524267-2823-4df6-91b0-314b96d3eb53,FLOW,43317aae-989f-4d14-8136-2820fac6a70c,4,-1,,-1,...,None,None,None,None,None,None,None,None,None,None
4,START,8119de7c-c6d0-4c68-89ea-a70511b2b48d,SysClient0060.systemia.com,b9029efa-5411-4d13-a8bd-982fcefab9e5,FLOW,4e9a0156-d8a0-499b-8ef0-c875fbd716df,4,-1,,-1,...,None,None,None,None,None,None,None,None,None,None


#### 2. EDA

In [4]:
total_rows = 0
total_malicious = 0


# Revisiting the malicious - benign distribution from before,

for file in files:
    data = pd.read_parquet(file, columns=["label"])

    total_rows += len(data)
    total_malicious += data["label"].sum()

total_benign = total_rows - total_malicious

print(f"Total events: {total_rows:,}")
print(f"Benign/unlabelled events: {total_benign:,}")
print(f"Malicious events: {total_malicious:,}")
print(f"Malicious percentage: {(total_malicious / total_rows) * 100:.3f}%")

Total events: 59,405,183
Benign/unlabelled events: 59,308,239
Malicious events: 96,944
Malicious percentage: 0.163%


### 3. Data Cleaning - Missing, Infinite and Duplicate Values

In [5]:
# Identify and calculate percentage of missing value

missing_counts = {}
total_rows = 0

for file in files:

    parquet_file = pq.ParquetFile(file)
    metadata = parquet_file.metadata

    total_rows += metadata.num_rows

    for i in range(metadata.num_columns):

        column_name = metadata.schema.column(i).name
        null_count = 0

        for row_group in range(metadata.num_row_groups):

            stats = metadata.row_group(row_group).column(i).statistics

            if stats is not None and stats.null_count is not None:
                null_count += stats.null_count

        missing_counts[column_name] = (
            missing_counts.get(column_name, 0) + null_count
        )

# Calculate percentage of missing values
missing_percentage = (
    pd.Series(missing_counts) / total_rows * 100
).sort_values(ascending=False)

print(missing_percentage)

requesting_domain      99.994997
requesting_user        99.994997
privileges             99.994844
user_name              99.994403
requesting_logon_id    99.994112
                         ...    
ppid                    0.000000
acuity_level            0.000000
id                      0.000000
object                  0.000000
actorID                 0.000000
Length: 62, dtype: float64


In [6]:
# drop columns which have more than 99% missing values
# I might undo this later
# But I will leave this for now, so as to manage data easily

high_missing_cols = missing_percentage[missing_percentage > 99].index.tolist()

print(f"Columns with >99% missing values: {len(high_missing_cols)}")
print(high_missing_cols)


Columns with >99% missing values: 22
['requesting_domain', 'requesting_user', 'privileges', 'user_name', 'requesting_logon_id', 'logon_id', 'task_pid', 'task_process_uuid', 'path', 'task_name', 'payload', 'context_info', 'sid', 'user', 'tgt_pid_uuid', 'type', 'value', 'data', 'key', 'new_path', 'start_time', 'end_time']


In [7]:
# Check and remove duplicate rows from each file

# total_duplicates = 0

# for file in files:
#     data = pd.read_parquet(file)

#     duplicates = data.duplicated().sum()
#     total_duplicates += duplicates

#     if duplicates > 0:
#         data = data.drop_duplicates()
#         data.to_parquet(file, index=False)

# print(f"Total duplicates removed: {total_duplicates:,}")

# From results, we see that there were no duplicates
# I commented it out to save RAM

In [8]:
# # Check for infinite values

# numeric_cols = data.select_dtypes(include=np.number).columns

# infinite_values = np.isinf(data[numeric_cols]).sum()

# print(infinite_values[infinite_values > 0])

# # From results, we see that there were no infinite values
# # I commented it out to save RAM

### 4. Create Data Sample (with referrence to temporal order)

In [9]:

# Specify the location where the sample will be stored
path = Path("/content/drive/MyDrive/solutions")
sample_path = path / "sample_cleaned"

sample_path.mkdir(parents=True, exist_ok=True)


# Select attack hosts and corresponding control hosts
selected_hosts = [
    # Attack hosts
    "sysclient0201",
    "sysclient0501",
    "sysclient0811",
    "sysclient0051",
    "sysclient0351",

    # Control hosts
    "sysclient0202",
    "sysclient0502",
    "sysclient0812",
    "sysclient0052",
    "sysclient0352"
]


# Get only the parquet files belonging to the selected hosts
selected_files = [
    file for file in files
    if any(host in file.name.lower() for host in selected_hosts)
]

print(f"Selected files: {len(selected_files)}")


# Specify the columns I no longer need
drop_cols = set(
    high_missing_cols +
    ["id", "actorID", "objectID"]
)


# Process the selected files one at a time to reduce RAM usage
for i, file in enumerate(selected_files, start=1):

    print(f"\n[{i}/{len(selected_files)}] Processing {file.name}")

    # Load one host at a time
    data = pd.read_parquet(file)

    # Drop highly missing and identifier columns
    data = data.drop(
        columns=drop_cols,
        errors="ignore"
    )

    # Convert timestamp to datetime
    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True
    )

    # Arrange each host's events in chronological order
    data = data.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    # Save each cleaned host separately
    output_file = sample_path / file.name

    data.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Saved {file.name} | "
        f"{len(data):,} rows | "
        f"{data.shape[1]} columns"
    )

    # Clear the current file from memory before loading the next
    del data


print("\nDone.")

Selected files: 12

[1/12] Processing 2019-09-24_AIA-501-525_sysclient0502.parquet
Saved 2019-09-24_AIA-501-525_sysclient0502.parquet | 4,246,693 rows | 34 columns

[2/12] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet | 136,241 rows | 37 columns

[3/12] Processing 2019-09-25_AIA-351-375_sysclient0352.parquet
Saved 2019-09-25_AIA-351-375_sysclient0352.parquet | 2,476,116 rows | 34 columns

[4/12] Processing 2019-09-24_AIA-801-825_sysclient0811.parquet
Saved 2019-09-24_AIA-801-825_sysclient0811.parquet | 3,818,669 rows | 34 columns

[5/12] Processing 2019-09-25_AIA-51-75_sysclient0052.parquet
Saved 2019-09-25_AIA-51-75_sysclient0052.parquet | 2,406,290 rows | 34 columns

[6/12] Processing 2019-09-23_AIA-201-225_sysclient0202.parquet
Saved 2019-09-23_AIA-201-225_sysclient0202.parquet | 4,706,585 rows | 34 columns

[7/12] Processing 2019-09-25_AIA-351-375_sysclient0351.parquet
Saved

### 5. Feature Dropping

In [10]:
# # I am dropping these are unique identifiers so as to avoid memorisation rather than behavioural learning.

# features_to_drop = [
#     "id",
#     "actorID",
#     "objectID"
# ]

# # I am also dropping the columns that have over 99% missing values from the previous check
# data = data.drop(columns=high_missing_cols)

NameError: name 'data' is not defined